## Game Control

Game Control measures a team's play-by-play dominance by calculating their time-weighted average score differential across a game. A positive rating indicates how many points on average a team led by throughout the contest duration:

$$\text{Game Control}_{\text{game}} = \frac{\sum (\text{Score Differential}_i \times \Delta t_i)}{\sum \Delta t_i}$$

$$\text{Season Game Control} = \frac{1}{N_{\text{games}}} \sum_{g=1}^{N_{\text{games}}} \text{Game Control}_g$$

Below we add the libraries we will need below.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import seaborn as sns
from IPython.display import display, Markdown
import nfl_data_py as nfl

Data Import & Processing

We pull official NFL play-by-play data from 2020 through 2025 using `nfl_data_py` and filter the data to get clean regular season data.

In [ ]:
seasons = list(range(2020, 2026))

# Fetch Play-by-Play data
pbp_raw = nfl.import_pbp_data(seasons)

# Filter completed plays and compute duration between play events
pbp_clean = pbp_raw[
    (pbp_raw['season_type'] == 'REG') &
    (pbp_raw['game_seconds_remaining'].notna()) &
    (pbp_raw['total_home_score'].notna()) &
    (pbp_raw['total_away_score'].notna()) &
    (pbp_raw['home_team'].notna()) &
    (pbp_raw['away_team'].notna())
].copy()

pbp_clean = pbp_clean.sort_values(['season', 'game_id', 'game_seconds_remaining'], ascending=[True, True, False])

pbp_clean['home_lead'] = pbp_clean['total_home_score'] - pbp_clean['total_away_score']

# Elapsed seconds from previous play state
prev_seconds = pbp_clean.groupby('game_id')['game_seconds_remaining'].shift(1).fillna(3600)
pbp_clean['play_duration'] = prev_seconds - pbp_clean['game_seconds_remaining']
pbp_clean['play_duration'] = np.maximum(0, pbp_clean['play_duration'])

### Game Control Calculation

Here is the actual code for game control calculations. We'll get home and away game control here so it is easier to get the data into a long form.

In [ ]:
# Calculate time-weighted score differential per game for home and away teams
def calc_game_gc(group):
    total_time = group['play_duration'].sum()
    denom = total_time if total_time > 0 else 1.0
    home_gc = (group['home_lead'] * group['play_duration']).sum() / denom
    return pd.Series({
        'total_time': total_time,
        'home_gc': home_gc,
        'away_gc': -home_gc
    })

game_control_by_game = pbp_clean.groupby(['season', 'game_id', 'home_team', 'away_team'], group_keys=False).apply(calc_game_gc).reset_index()

# Reshape data into long team-game format
home_df = game_control_by_game[['season', 'game_id', 'home_team', 'home_gc']].rename(columns={'home_team': 'team', 'home_gc': 'game_control'})
away_df = game_control_by_game[['season', 'game_id', 'away_team', 'away_gc']].rename(columns={'away_team': 'team', 'away_gc': 'game_control'})

team_games = pd.concat([home_df, away_df], ignore_index=True)

# Aggregate across each season
season_game_control = team_games.groupby(['season', 'team']).agg(
    games_played=('game_control', 'count'),
    avg_game_control=('game_control', lambda x: np.round(x.mean(), 2))
).reset_index().sort_values(['season', 'avg_game_control'], ascending=[True, False])

Here are the Top 10 Game Control Teams by Season

In [ ]:
for yr in sorted(season_game_control['season'].unique()):
    top_10 = (
        season_game_control[season_game_control['season'] == yr]
        .head(10)
        .reset_index(drop=True)
    )
    top_10['Rank'] = top_10.index + 1
    top_10 = top_10[['Rank', 'team', 'games_played', 'avg_game_control']].rename(columns={
        'team': 'Team',
        'games_played': 'Games Played',
        'avg_game_control': 'Avg Game Control (Points)'
    })
    
    display(Markdown(f"### Season: {yr}"))
    display(Markdown(top_10.to_markdown(index=False)))

### Adjusted Game Control

Below we calculate the adjusted game control which adjusts for the opponent as well as whether a team is home or away.

Note that code here for calculating the adjusted game score. We first build a matrix with all zeros. The number of rows is the number of games and the number of columns is the number of teams. We then replace some of the zeroes in a row with a $1$ for the home team of a game and a $-1$ for the away team of the game. We also add an intercept term to the matrix which is just a column of ones.

In [ ]:
def calc_adj_gc_scratch(df):
    teams = np.unique(np.concatenate([df['home_team'].unique(), df['away_team'].unique()]))
    n_games = len(df)
    season = df['season'].iloc[0]
    
    # Construct design matrix: Home (+1), Away (-1)
    X = pd.DataFrame(0.0, index=range(n_games), columns=teams)
    
    for i, (_, row) in enumerate(df.iterrows()):
        X.loc[i, row['home_team']] = 1.0
        X.loc[i, row['away_team']] = -1.0
        
    margin = df['home_gc'].values
    
    # Add intercept column (column of ones)
    X_design = sm.add_constant(X)
    #X_design=X   
    # Fit OLS model
    fit = sm.OLS(margin, X_design).fit()
    srs_vec = fit.params.fillna(0)
    srs_vec = srs_vec.rename(index={'const': '(Intercept)'})
    
    # Center parameters
    srs_centered = srs_vec - srs_vec.mean()
    
    return pd.DataFrame({
        'team': srs_centered.index,
        'season': season,
        'Adj_GC': np.round(srs_centered.values, 2)
    })

This next set of code is some slickness to run the function above by season and bind the resulting output.

In [ ]:
ratings_list = []
for yr, season_df in game_control_by_game.groupby('season'):
    ratings_list.append(calc_adj_gc_scratch(season_df))

ratings_by_season = pd.concat(ratings_list, ignore_index=True)

Below we get the sorted values in the model.

In [ ]:
for yr in sorted(ratings_by_season['season'].unique()):
    top_10 = (
        ratings_by_season[ratings_by_season['season'] == yr]
        .sort_values(by='Adj_GC', ascending=False)
        .reset_index(drop=True)
    )
    top_10['Rank'] = top_10.index + 1
    
    top_10 = top_10[['Rank', 'season', 'team', 'Adj_GC']].rename(columns={
        'season': 'season',
        'team': 'Team',
        'Adj_GC': 'Adj_game_control'
    })
    
    display(Markdown(f"### Season: {yr}"))
    display(Markdown(top_10.to_markdown(index=False)))

### Task 0 Answer

**What does the (Intercept) represent for these data?**

The **(Intercept)** represents the overall **Home Field Advantage** measured in Game Control points across all games in a given season[cite: 3]. Because home teams are encoded as $+1$ and away teams as $-1$, the intercept captures the average baseline margin/lead held by home teams simply by playing at home[cite: 3].

With the code below, we want to investigate the correlation between Average game control and adjusted game control. First we join the data and drop the intercept terms.

In [ ]:
all_df = season_game_control.merge(
    ratings_by_season, on=['season', 'team'], how='right'
)
all_df = all_df[all_df['team'] != '(Intercept)'].reset_index(drop=True)

Make a plot with labels for the plotting points.

In [ ]:
plt.figure(figsize=(10, 8))

# Quadrant baseline reference lines
plt.axhline(0, linestyle='--', color='gray', alpha=0.7, linewidth=0.8)
plt.axvline(0, linestyle='--', color='gray', alpha=0.7, linewidth=0.8)

# 1:1 Parity line
min_val = min(all_df['avg_game_control'].min(), all_df['Adj_GC'].min()) - 1
max_val = max(all_df['avg_game_control'].max(), all_df['Adj_GC'].max()) + 1
plt.plot([min_val, max_val], [min_val, max_val], linestyle=':', color='firebrick', alpha=0.7, label='1:1 Parity')

sns.scatterplot(
    data=all_df,
    x='avg_game_control',
    y='Adj_GC',
    hue='season',
    palette='tab10',
    alpha=0.8,
    s=60
)

for _, row in all_df.iterrows():
    plt.text(row['avg_game_control'] + 0.1, row['Adj_GC'] + 0.1, row['team'], fontsize=7, alpha=0.7)

plt.title("NFL Raw vs. Opponent-Adjusted Game Control", fontsize=14, fontweight='bold')
plt.suptitle("Teams above the red dotted line benefited from tough schedule adjustments", fontsize=10, y=0.92, color='gray')
plt.xlabel("Raw Average Game Control (Points Lead/Trail)", fontweight='bold')
plt.ylabel("Adjusted Game Control (Points Lead/Trail)", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

Make the same plot but put the team colors as the plot points instead of the logos.

In [ ]:
try:
    team_desc = nfl.import_team_desc()
    color_dict = dict(zip(team_desc['team_abbr'], team_desc['team_color']))
except Exception:
    color_dict = {}

plt.figure(figsize=(10, 8))
plt.axhline(0, linestyle='--', color='gray', alpha=0.7)
plt.axvline(0, linestyle='--', color='gray', alpha=0.7)
plt.plot([min_val, max_val], [min_val, max_val], linestyle=':', color='firebrick')

for _, row in all_df.iterrows():
    team_color = color_dict.get(row['team'], '#1f77b4')
    plt.scatter(row['avg_game_control'], row['Adj_GC'], color=team_color, s=50, alpha=0.85)

plt.title("NFL Game Control: Raw vs. Adjusted", fontsize=14, fontweight='bold')
plt.xlabel("Raw Game Control (Points)", fontweight='bold')
plt.ylabel("Adjusted Game Control (Points)", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


### Task

1. Find the correlation by season between the average game control and the adjusted game control.

2. Suppose we build these models for FBS college football teams?  Do you think the correlations that you calculated in the previous task would be similar?  Why or why not?